In [1]:
!pip install -q transformers datasets evaluate accelerate seqeval

from datasets import load_dataset

dataset = load_dataset('wikiann', 'en')
print(dataset)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.0 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

en/validation-00000-of-00001.parquet:   0%|          | 0.00/748k [00:00<?, ?B/s]

en/test-00000-of-00001.parquet:   0%|          | 0.00/748k [00:00<?, ?B/s]

en/train-00000-of-00001.parquet:   0%|          | 0.00/1.50M [00:00<?, ?B/s]

Generating validation split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/20000 [00:00<?, ? examples/s]

DatasetDict({
    validation: Dataset({
        features: ['tokens', 'ner_tags', 'langs', 'spans'],
        num_rows: 10000
    })
    test: Dataset({
        features: ['tokens', 'ner_tags', 'langs', 'spans'],
        num_rows: 10000
    })
    train: Dataset({
        features: ['tokens', 'ner_tags', 'langs', 'spans'],
        num_rows: 20000
    })
})


In [2]:
# Get label names
label_names = dataset['train'].features['ner_tags'].feature.names
print('Labels:', label_names)

Labels: ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC']


In [3]:
# Look at one example — like an annotation view
example = dataset['train'][0]

for token, tag in zip(example['tokens'], example['ner_tags']):
    print(f"{token:20} {label_names[tag]}")

R.H.                 B-ORG
Saunders             I-ORG
(                    O
St.                  B-ORG
Lawrence             I-ORG
River                I-ORG
)                    O
(                    O
968                  O
MW                   O
)                    O


In [4]:
from collections import Counter

all_tags = [tag for row in dataset['train']['ner_tags'] for tag in row]
counts = Counter(all_tags)

print(f"{'Label':12} {'Count':>8}  {'%':>6}")
print('-' * 30)
for tag_id, count in sorted(counts.items()):
    print(f"{label_names[tag_id]:12} {count:>8}  {count/len(all_tags)*100:>5.1f}%")

Label           Count       %
------------------------------
O               81362   50.7%
B-PER            9164    5.7%
I-PER           14698    9.2%
B-ORG            9422    5.9%
I-ORG           23226   14.5%
B-LOC            9345    5.8%
I-LOC           13177    8.2%


In [5]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize_and_align(examples):
    tokenized = tokenizer(
        examples["tokens"], truncation=True, is_split_into_words=True
    )
    all_labels = []
    for i, labels in enumerate(examples["ner_tags"]):
        word_ids = tokenized.word_ids(batch_index=i)
        aligned = []
        prev = None
        for word_id in word_ids:
            if word_id is None:
                aligned.append(-100)      # special tokens → ignored in loss
            elif word_id != prev:
                aligned.append(labels[word_id])
            else:
                aligned.append(-100)      # subword continuation → ignored
            prev = word_id
        all_labels.append(aligned)
    tokenized["labels"] = all_labels
    return tokenized

tokenized_dataset = dataset.map(tokenize_and_align, batched=True)
print("Done!", tokenized_dataset)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Done! DatasetDict({
    validation: Dataset({
        features: ['tokens', 'ner_tags', 'langs', 'spans', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 10000
    })
    test: Dataset({
        features: ['tokens', 'ner_tags', 'langs', 'spans', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 10000
    })
    train: Dataset({
        features: ['tokens', 'ner_tags', 'langs', 'spans', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 20000
    })
})


In [6]:
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(label_names),
    id2label={i: l for i, l in enumerate(label_names)},
    label2id={l: i for i, l in enumerate(label_names)}
)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [7]:
import evaluate
import numpy as np

seqeval = evaluate.load("seqeval")

def compute_metrics(eval_preds):
    logits, labels = eval_preds
    predictions = np.argmax(logits, axis=-1)
    true_labels = [
        [label_names[l] for l in label if l != -100]
        for label in labels
    ]
    true_preds = [
        [label_names[p] for p, l in zip(pred, label) if l != -100]
        for pred, label in zip(predictions, labels)
    ]
    results = seqeval.compute(predictions=true_preds, references=true_labels)
    return {"f1": results["overall_f1"], "accuracy": results["overall_accuracy"]}

In [8]:
from transformers import TrainingArguments, Trainer, DataCollatorForTokenClassification

args = TrainingArguments(
    output_dir="ner-results",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
)

data_collator = DataCollatorForTokenClassification(tokenizer)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

Epoch,Training Loss,Validation Loss,F1,Accuracy
1,0.298035,0.255336,0.804706,0.923214
2,0.209049,0.245301,0.814766,0.928616
3,0.156291,0.250301,0.821593,0.930503


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3750, training_loss=0.24705401407877603, metrics={'train_runtime': 455.0376, 'train_samples_per_second': 131.857, 'train_steps_per_second': 8.241, 'total_flos': 462214875911712.0, 'train_loss': 0.24705401407877603, 'epoch': 3.0})

In [9]:
results = trainer.evaluate()
print(f"F1 Score:  {results['eval_f1']:.4f}")
print(f"Accuracy:  {results['eval_accuracy']:.4f}")
print(f"Loss:      {results['eval_loss']:.4f}")

F1 Score:  0.8216
Accuracy:  0.9305
Loss:      0.2503


In [10]:
from transformers import pipeline

ner = pipeline("ner", model=model, tokenizer=tokenizer, aggregation_strategy="simple")

sentence = "Abraham Lincoln was born in Larue County, KY and Lincoln Memorial in Washington D.C. was built for him by Henry Bacon."
results_ner = ner(sentence)

for entity in results_ner:
    print(f"{entity['word']:20} {entity['entity_group']:8} {entity['score']:.2f}")

abraham lincoln      PER      0.90
larue county, ky     LOC      0.84
lincoln memorial     ORG      0.95
washington d. c      LOC      0.56
.                    ORG      0.51
henry bacon          PER      0.97


In [11]:
# Run model on the full test set
predictions = trainer.predict(tokenized_dataset["test"])
pred_labels = predictions.predictions.argmax(-1)
true_labels = predictions.label_ids

print("Predictions ready!")
print(f"Test set size: {len(pred_labels)} sentences")

Predictions ready!
Test set size: 10000 sentences


In [12]:
errors = {"wrong_type": [], "false_positive": [], "false_negative": []}

for sent_idx in range(len(true_labels)):
    tokens = dataset["test"][sent_idx]["tokens"]
    true = [label_names[l] for l in true_labels[sent_idx] if l != -100]
    pred = [label_names[p] for p, l in zip(pred_labels[sent_idx], true_labels[sent_idx]) if l != -100]

    for i, (t, p) in enumerate(zip(true, pred)):
        token = tokens[i] if i < len(tokens) else "?"
        if t == p:
            continue
        if t == "O" and p != "O":
            errors["false_positive"].append({"token": token, "predicted": p})
        elif t != "O" and p == "O":
            errors["false_negative"].append({"token": token, "true": t})
        elif t != "O" and p != "O" and t != p:
            errors["wrong_type"].append({"token": token, "true": t, "predicted": p})

print(f"Wrong type:      {len(errors['wrong_type'])}")
print(f"False positives: {len(errors['false_positive'])}")
print(f"False negatives: {len(errors['false_negative'])}")

Wrong type:      4211
False positives: 698
False negatives: 769


In [13]:
from collections import Counter

# Most common wrong type confusions
confusions = Counter(
    (e["true"], e["predicted"]) for e in errors["wrong_type"]
)

print(f"{'True':12} {'Predicted':12} {'Count':>6}")
print("-" * 32)
for (true, pred), count in confusions.most_common(10):
    print(f"{true:12} {pred:12} {count:>6}")

True         Predicted     Count
--------------------------------
I-LOC        I-ORG           685
I-ORG        I-LOC           594
I-PER        I-ORG           515
I-ORG        I-PER           506
B-ORG        B-LOC           402
B-LOC        B-ORG           297
B-ORG        B-PER           287
B-PER        B-ORG           220
B-ORG        I-ORG           119
B-LOC        I-LOC           102


In [14]:
# Find real sentence examples for top confusions
def find_examples(true_label, pred_label, n=3):
    examples = []
    for sent_idx in range(len(true_labels)):
        tokens = dataset["test"][sent_idx]["tokens"]
        true = [label_names[l] for l in true_labels[sent_idx] if l != -100]
        pred = [label_names[p] for p, l in zip(pred_labels[sent_idx], true_labels[sent_idx]) if l != -100]
        for i, (t, p) in enumerate(zip(true, pred)):
            if t == true_label and p == pred_label and i < len(tokens):
                examples.append({
                    "token": tokens[i],
                    "sentence": " ".join(tokens)
                })
        if len(examples) >= n:
            break
    return examples

# Show examples for top 3 confusions
for true_label, pred_label in [("I-LOC", "I-ORG"), ("I-PER", "I-ORG"), ("B-ORG", "B-LOC")]:
    print(f"\n{true_label} → {pred_label}:")
    for ex in find_examples(true_label, pred_label):
        print(f"  '{ex['token']}' in: {ex['sentence'][:80]}")


I-LOC → I-ORG:
  'Network' in: Its closest competitor is the Encash Network Service .
  'Service' in: Its closest competitor is the Encash Network Service .
  'Castle' in: *Seated figure of Sir Walter Scott for Powderham Castle ( 1832 )

I-PER → I-ORG:
  'of' in: List of The O.C. characters
  'The' in: List of The O.C. characters
  'O.C.' in: List of The O.C. characters
  'characters' in: List of The O.C. characters

B-ORG → B-LOC:
  'Jhansi' in: Goes to Jhansi ( JHS )
  'Crested' in: Crested caracara , ''Caracara cheriway '' ( A )
  'Mittani' in: ' '' Mittani '' '


In [15]:
print("=== ERROR ANALYSIS SUMMARY ===\n")
print(f"Total test tokens evaluated: {sum(len(s) for s in dataset['test']['tokens'])}")
print(f"Wrong type errors:  {len(errors['wrong_type'])} ({len(errors['wrong_type'])/(len(errors['wrong_type'])+len(errors['false_positive'])+len(errors['false_negative']))*100:.1f}% of all errors)")
print(f"False positives:    {len(errors['false_positive'])} ({len(errors['false_positive'])/(len(errors['wrong_type'])+len(errors['false_positive'])+len(errors['false_negative']))*100:.1f}% of all errors)")
print(f"False negatives:    {len(errors['false_negative'])} ({len(errors['false_negative'])/(len(errors['wrong_type'])+len(errors['false_positive'])+len(errors['false_negative']))*100:.1f}% of all errors)")
print(f"\nTop confusion: LOC↔ORG ({685+594} cases) — geopolitical entity ambiguity")
print(f"Dataset errors found: model predictions arguably more correct than ground truth labels in some cases")

=== ERROR ANALYSIS SUMMARY ===

Total test tokens evaluated: 80326
Wrong type errors:  4211 (74.2% of all errors)
False positives:    698 (12.3% of all errors)
False negatives:    769 (13.5% of all errors)

Top confusion: LOC↔ORG (1279 cases) — geopolitical entity ambiguity
Dataset errors found: model predictions arguably more correct than ground truth labels in some cases
